In [ ]:
import os
import argparse
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import label

In [ ]:
def preprocess(img: np.ndarray,
               brightness: float = 1.3,
               clahe_clip: float = 2.0,
               denoise_h: int = 20) -> np.ndarray:
    # stretch to [0,255]
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    # global brightness / contrast knob
    img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)
    # local contrast equalisation
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    img = clahe.apply(img)
    # fast non‑local means denoise (good on speckle)
    img = cv2.fastNlMeansDenoising(img, None, h=denoise_h,
                                   templateWindowSize=7, searchWindowSize=21)
    return img

In [ ]:
def detect_objects_advanced(tile: np.ndarray,
                            area_thresh: float = 0.001,
                            intensity_ratio: float = 2.0):
    pre = preprocess(tile)

    # adaptive threshold tuned for speckle
    bin_img = cv2.adaptiveThreshold(pre, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 51, -5)

    # morphology to remove pepper noise and fill holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, kernel, iterations=1)
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_CLOSE, kernel, iterations=2)

    labeled, num = label(bin_img)
    objs = []
    min_pixels = int(area_thresh * tile.shape[0] * tile.shape[1])
    background_median = np.median(tile)

    for i in range(1, num + 1):
        ys, xs = np.where(labeled == i)
        if xs.size < min_pixels:
            continue
        if tile[ys, xs].mean() < intensity_ratio * background_median:
            continue
        x1, x2 = xs.min(), xs.max()
        y1, y2 = ys.min(), ys.max()
        objs.append((x1, y1, x2, y2))
    return objs

In [ ]:

def split_image(image: np.ndarray, max_tile_size=(2000, 2000)):
    h, w = image.shape
    if h <= max_tile_size[1] and w <= max_tile_size[0]:
        return [(image, 0, 0)]
    tile_w = min(w, max_tile_size[0])
    tile_h = min(h, max_tile_size[1])
    tiles = []
    for y in range(0, h, tile_h):
        for x in range(0, w, tile_w):
            tiles.append((image[y:y + tile_h, x:x + tile_w], x, y))
    return tiles

In [ ]:
def process_image_adv(image_path: str,
                      max_tile_size=(2000, 2000),
                      area_thresh: float = 0.001,
                      intensity_ratio: float = 2.0):
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8),
                       cv2.IMREAD_GRAYSCALE)
    tiles = split_image(img, max_tile_size)
    all_objects = []
    for tile, x_off, y_off in tiles:
        objs = detect_objects_advanced(tile, area_thresh, intensity_ratio)
        for x1, y1, x2, y2 in objs:
            all_objects.append((x1 + x_off, y1 + y_off,
                                x2 + x_off, y2 + y_off))
    return all_objects


def draw_objects(image_path: str, objects, output_path: str):
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8),
                       cv2.IMREAD_COLOR)
    for x1, y1, x2, y2 in objects:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imwrite(output_path, img)


def process_folder(input_folder: str,
                   output_folder: str,
                   max_tile_size=(2000, 2000),
                   area_thresh: float = 0.001,
                   intensity_ratio: float = 2.0):
    os.makedirs(output_folder, exist_ok=True)
    summary = []
    for fname in os.listdir(input_folder):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        inp = os.path.join(input_folder, fname)
        out = os.path.join(output_folder,
                           f"{os.path.splitext(fname)[0]}_out.jpg")
        objs = process_image_adv(inp, max_tile_size,
                                 area_thresh, intensity_ratio)
        draw_objects(inp, objs, out)
        summary.append({"image": fname,
                        "objects_detected": len(objs),
                        "output_image": out})
        print(f"Processed {fname}: {len(objs)} objects -> {out}")

    # optional CSV summary
    if summary:
        df = pd.DataFrame(summary)
        df.to_csv(os.path.join(output_folder, "summary.csv"), index=False)
        print(f"CSV summary saved to {output_folder}/summary.csv")
